# 09 - Scaling Analysis

Plot legibility score L as a function of model parameter count.
Compare with Lanham et al.'s inverse-scaling faithfulness finding.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

def model_tag(name):
    return name.split("/")[-1].lower().replace("-", "_")

def get_acc(condition):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    return np.mean(results) if results else None

def bootstrap_ci(arr, n=10000, seed=42):
    rng = np.random.RandomState(seed)
    boots = sorted([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)])
    return boots[int(0.025*n)], boots[int(0.975*n)]

# Parameter counts (approximate)
param_counts = {
    "Qwen/Qwen3-1.7B": 1.7,
    "Qwen/Qwen3-4B": 4.0,
    "Qwen/Qwen3-8B": 8.0,
    "Qwen/Qwen3-14B": 14.0,
}

In [ ]:
# Compute L for each model size
scaling_results = []

for model_name in SCALING_MODELS:
    tag = model_tag(model_name)
    params = param_counts.get(model_name, 0)

    # For 4B, use core conditions (no tag suffix)
    if model_name == PRIMARY_MODEL:
        nc = get_acc("no_cot")
        sp = get_acc("self_prefill")
        cp = get_acc("cross_prefill")
        ps = get_acc("paraphrase_self")
        pc = get_acc("paraphrase_cross")
        normal = get_acc("normal")
    else:
        nc = get_acc(f"no_cot_{tag}")
        sp = get_acc(f"self_prefill_{tag}")
        cp = get_acc(f"cross_prefill_{tag}")
        ps = get_acc(f"paraphrase_self_{tag}")
        pc = get_acc(f"paraphrase_cross_{tag}")
        normal = None  # Not computed for scaling models

    total_value = (sp - nc) if (sp is not None and nc is not None) else None
    L = ((pc - nc) / total_value) if (pc is not None and total_value and total_value > 0.01) else None

    row = {
        "model": model_name, "params_b": params, "tag": tag,
        "no_cot": nc, "self_prefill": sp, "cross_prefill": cp,
        "paraphrase_self": ps, "paraphrase_cross": pc,
        "normal": normal, "total_value": total_value, "L": L,
    }
    scaling_results.append(row)
    print(f"{model_name:25s} | no_cot={nc} sp={sp} cp={cp} ps={ps} pc={pc} | L={L}")

# Save
with open(RESULTS_DIR / "scaling_results.json", "w") as f:
    json.dump(scaling_results, f, indent=2, default=str)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy by model size
valid = [r for r in scaling_results if r["self_prefill"] is not None]
params = [r["params_b"] for r in valid]
for cond, color, label in [
    ("no_cot", "gray", "No COT"),
    ("self_prefill", "#8e44ad", "Self prefill"),
    ("cross_prefill", "#9b59b6", "Cross prefill"),
    ("paraphrase_self", "#2980b9", "Paraphrase self"),
    ("paraphrase_cross", "#27ae60", "Paraphrase cross"),
]:
    vals = [r[cond] for r in valid]
    if any(v is not None for v in vals):
        plot_vals = [v if v is not None else np.nan for v in vals]
        ax1.plot(params, plot_vals, "o-", color=color, label=label, markersize=8)

ax1.set_xlabel("Parameters (B)")
ax1.set_ylabel("Accuracy")
ax1.set_title("Accuracy vs Model Size", fontweight="bold")
ax1.legend(fontsize=8)
ax1.set_ylim(0, 1)

# Right: Legibility L by model size
l_vals = [r["L"] for r in valid]
l_valid = [(p, l) for p, l in zip(params, l_vals) if l is not None]
if l_valid:
    lp, ll = zip(*l_valid)
    ax2.plot(lp, ll, "o-", color="#27ae60", markersize=10, linewidth=2)
    for x, y in zip(lp, ll):
        ax2.annotate(f"{y:.3f}", (x, y), textcoords="offset points",
                     xytext=(0, 12), ha="center", fontsize=10)

ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3, label="Perfect legibility")
ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.3, label="Monitoring threshold")
ax2.set_xlabel("Parameters (B)")
ax2.set_ylabel("Legibility score (L)")
ax2.set_title("Legibility vs Model Size", fontweight="bold")
ax2.set_ylim(0, 1.5)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "scaling_legibility.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table
import pandas as pd

df = pd.DataFrame(scaling_results)
display_cols = ["model", "params_b", "no_cot", "self_prefill", "cross_prefill",
                "paraphrase_self", "paraphrase_cross", "total_value", "L"]
print(df[display_cols].to_string(index=False, float_format="{:.3f}".format))